In [2]:
import os
import pandas as pd
import psycopg2
from psycopg2 import sql

# ---------- CONFIGURATION ----------
CSV_FOLDER = r"C:\Users\leedf\weatherData"  # Folder containing your CSV files
DB_NAME = "weather"
DB_USER = "postgres"
DB_PASSWORD = "071726postGRES"
DB_HOST = "localhost"
DB_PORT = "5432"

# ---------- CONNECT TO POSTGRES ----------
try:
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    conn.autocommit = True
    cursor = conn.cursor()
    print("Connected to PostgreSQL.")
except Exception as e:
    raise SystemExit(f"Database connection failed: {e}")

# ---------- PROCESS EACH CSV ----------
for file in os.listdir(CSV_FOLDER):
    if file.endswith(".csv"):
        table_name = os.path.splitext(file)[0]  # Table name from file name
        file_path = os.path.join(CSV_FOLDER, file)

        # Load CSV into DataFrame
        df = pd.read_csv(file_path)

        # Validate required columns
        required_cols = {"zip", "state", "city"}
        if not required_cols.issubset(df.columns):
            print(f"Skipping {file}: Missing required columns.")
            continue

        # Create table SQL dynamically
        col_defs = []
        for col in df.columns:
            if col == "zip":
                col_defs.append(sql.SQL("{} INTEGER PRIMARY KEY").format(sql.Identifier(col)))
            else:
                col_defs.append(sql.SQL("{} TEXT").format(sql.Identifier(col)))

        create_table_query = sql.SQL("CREATE TABLE IF NOT EXISTS {} ({});").format(
            sql.Identifier(table_name),
            sql.SQL(", ").join(col_defs)
        )

        try:
            cursor.execute(create_table_query)
            print(f"Table '{table_name}' created.")
        except Exception as e:
            print(f"Error creating table {table_name}: {e}")
            continue

        # Insert data
        for _, row in df.iterrows():
            insert_query = sql.SQL("INSERT INTO {} ({}) VALUES ({}) ON CONFLICT (zip) DO NOTHING;").format(
                sql.Identifier(table_name),
                sql.SQL(", ").join(map(sql.Identifier, df.columns)),
                sql.SQL(", ").join(sql.Placeholder() * len(df.columns))
            )
            try:
                cursor.execute(insert_query, tuple(row))
            except Exception as e:
                print(f"Insert error in {table_name}: {e}")

print("All CSV files processed.")

# ---------- CLEANUP ----------
cursor.close()
conn.close()


Connected to PostgreSQL.
Table 'us_average_humidity_by_zip' created.
Table 'us_average_precipitation_by_zip' created.
Table 'us_average_snowfall_by_zip' created.
Table 'us_average_temperature_by_zip' created.
All CSV files processed.
